# 🤖 Super AI po Polsku
### Asystent oparty na Claude Opus — rozumie Cię i odpowiada po polsku

**Funkcje:**
- Pełna rozmowa wieloturowa (pamięta kontekst)
- Odpowiedzi strumieniowane w czasie rzeczywistym
- Automatyczne wykrywanie nastroju wiadomości 🎭
- Podsumowanie rozmowy na żądanie

In [ ]:
# Instalacja zależności (uruchom raz)
!pip install anthropic --quiet

In [ ]:
import anthropic
import os
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

# Klucz API — ustaw zmienną środowiskową ANTHROPIC_API_KEY lub wpisz tutaj
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

SYSTEM_PROMPT = """\
Jesteś Super AI — inteligentnym, pomocnym i przyjaznym asystentem mówiącym wyłącznie po polsku.
Rozumiesz język polski doskonale: slang, skróty, regionalizmy, a nawet literówki.
Odpowiadasz zawsze po polsku, kulturalnie i rzeczowo.
Potrafisz pomagać w nauce, pisaniu, programowaniu, codziennych pytaniach i kreatywnych zadaniach.
Jeśli użytkownik napisze po angielsku, odpowiedz po polsku i delikatnie zasugeruj, że możesz rozmawiać po polsku.
Bądź naturalny — jak mądry znajomy, nie suchy robot.
"""

historia_rozmowy = []
print("✅ Klient API gotowy. Model:", MODEL)

In [ ]:
NASTROJE = {
    "pozytywny": "😊",
    "negatywny": "😟",
    "neutralny": "😐",
    "pytający": "🤔",
    "zaskoczony": "😲",
    "żartobliwy": "😄",
}

def wykryj_nastoj(tekst: str) -> str:
    """Szybka analiza nastroju wiadomości (jeden krótki call API)."""
    odpowiedz = client.messages.create(
        model=MODEL,
        max_tokens=10,
        system="Klasyfikuj nastrój wiadomości jednym słowem po polsku spośród: pozytywny, negatywny, neutralny, pytający, zaskoczony, żartobliwy. Odpowiedz TYLKO tym jednym słowem.",
        messages=[{"role": "user", "content": tekst}],
    )
    nastoj = odpowiedz.content[0].text.strip().lower()
    emotka = NASTROJE.get(nastoj, "💬")
    return f"{emotka} {nastoj}"


def chat(wiadomosc: str, pokaz_nastoj: bool = True) -> str:
    """Wyślij wiadomość i odbierz odpowiedź strumieniowaną."""
    global historia_rozmowy

    if pokaz_nastoj:
        nastoj = wykryj_nastoj(wiadomosc)
        print(f"\n🎭 Wykryty nastrój: {nastoj}")

    historia_rozmowy.append({"role": "user", "content": wiadomosc})

    print("\n🤖 Super AI:", flush=True)
    pelna_odpowiedz = ""

    with client.messages.stream(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        thinking={"type": "adaptive"},
        messages=historia_rozmowy,
    ) as strumien:
        for fragment in strumien.text_stream:
            print(fragment, end="", flush=True)
            pelna_odpowiedz += fragment

    print()  # nowa linia po zakończeniu
    historia_rozmowy.append({"role": "assistant", "content": pelna_odpowiedz})
    return pelna_odpowiedz


def podsumuj_rozmowe() -> None:
    """Generuje krótkie podsumowanie dotychczasowej rozmowy."""
    if not historia_rozmowy:
        print("Brak historii do podsumowania.")
        return

    msgs = historia_rozmowy + [{"role": "user", "content": "Podsumuj naszą rozmowę w 3-5 zdaniach po polsku."}]
    odpowiedz = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=SYSTEM_PROMPT,
        messages=msgs,
    )
    print("\n📋 Podsumowanie rozmowy:")
    print(odpowiedz.content[0].text)


def wyczysc_history() -> None:
    global historia_rozmowy
    historia_rozmowy = []
    print("🗑️ Historia wyczyszczona. Zaczynamy od nowa!")


print("✅ Funkcje załadowane.")

## Rozpocznij rozmowę

Wpisz swoją wiadomość w komórce poniżej i uruchom ją (`Shift+Enter`).

In [ ]:
chat("Cześć! Kim jesteś i co potrafisz?")

In [ ]:
# Napisz cokolwiek tutaj:
chat("Opowiedz mi coś ciekawego o Polsce.")

In [ ]:
# Kolejna wiadomość — AI pamięta cały kontekst rozmowy:
chat("A co sądzisz o polskiej kuchni?")

In [ ]:
# Podsumowanie całej rozmowy:
podsumuj_rozmowe()

In [ ]:
# Wyczyść historię i zacznij nową rozmowę:
wyczysc_history()

---
## Interaktywny czat (widżety)

Poniżej znajdziesz wersję z graficznym interfejsem — pole tekstowe i przycisk `Wyślij`.

In [ ]:
pole_tekstu = widgets.Text(
    placeholder="Napisz wiadomość po polsku...",
    layout=widgets.Layout(width="70%"),
)
przycisk = widgets.Button(description="Wyślij 💬", button_style="primary")
wyjscie = widgets.Output()

def na_klikniecie(_):
    tekst = pole_tekstu.value.strip()
    if not tekst:
        return
    pole_tekstu.value = ""
    with wyjscie:
        print(f"\n👤 Ty: {tekst}")
        chat(tekst)

przycisk.on_click(na_klikniecie)
display(widgets.HBox([pole_tekstu, przycisk]), wyjscie)